# 11 · Preview ClinicalTrials CSV Datasets

This notebook streams only the first few records from each large CSV inside `Clinical.zip`. It does **not** load 100+ MB of CSVs into memory or incorrectly index trial metadata as prose. Use further data profiling only if required for the clinical recommender.

In [ ]:
%run 00_foundation.ipynb
import zipfile, csv, io, json
ARCHIVE=ROOT/'data'/'course_sources'/'Clinical.zip'
if not ARCHIVE.exists(): raise FileNotFoundError(f'Missing {ARCHIVE}')

results=[]
with zipfile.ZipFile(ARCHIVE) as z:
    for info in z.infolist():
        if not info.filename.lower().endswith('.csv'):continue
        # Limit memory and runtime: these ClinicalTrials source files are up to 74 MB.
        with z.open(info) as stream, io.TextIOWrapper(stream,encoding='utf-8-sig',errors='replace',newline='') as handle:
            reader=csv.reader(handle)
            header=next(reader,[])
            preview=[]
            for i,row in enumerate(reader):
                if i>=3: break
                preview.append({header[j]:v[:120] for j,v in enumerate(row[:min(len(header),8)])})
        results.append({'member':info.filename,'size_MB':round(info.file_size/2**20,1),
                        'column_count':len(header),'first_columns':header[:14],
                        'example_rows_first_eight_fields':preview,
                        'row_count':'not computed (streamed preview only)',
                        'kind':'clinical trials structured records, not device specs'})
print(json.dumps(results,indent=2,ensure_ascii=False)[:13000])
